# 🍐 Little Fig Studio

**Train any LLM on any hardware.** Select your model, configure, launch.

| What | How |
|---|---|
| Quantization | FigQuant INT4 (beats NF4 on 156/156 layers) |
| Speed | 7× faster than BnB NF4 on GPU |
| Memory | Train 1.1B models in <4GB VRAM |
| Optimizer | FigMeZO (−18.6% loss, original research) |

**Run cells below ↓**

In [ ]:
# Install
!pip install -q torch
!git clone https://github.com/ticketguy/littlefig.git /content/littlefig --quiet 2>/dev/null || true
!cd /content/littlefig && pip install -q -e ".[train]"
!pip install -q uvicorn fastapi python-multipart

import torch, sys
sys.path.insert(0, '/content/littlefig/src')
print(f'\n✅ Ready | PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Launch Little Fig Studio UI
import subprocess, time, threading
from google.colab import output

# Start server in background
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'little_fig.web.server:app',
     '--host', '0.0.0.0', '--port', '8888'],
    cwd='/content/littlefig/src',
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

# Serve via Colab's built-in proxy
output.serve_kernel_port_as_window(8888)
print('\n🍐 Little Fig Studio launched!')
print('   A new tab/window should have opened with the UI.')
print('   If not, click the link above.')
print('\n   Keep this cell running to keep the server alive.')

---
## Or use Python directly (no UI)

Change `MODEL` to any HuggingFace model you want to train.

In [ ]:
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig

# === CHANGE THIS TO YOUR MODEL ===
MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# Other options:
# MODEL = 'google/gemma-3-4b-it'
# MODEL = 'Qwen/Qwen2.5-1.5B'
# MODEL = 'microsoft/phi-2'

model = FigModel.from_pretrained(MODEL, lora_r=16, lora_alpha=32, shared_codebook=True)
print(f'\n✅ Loaded {MODEL}')
print(f'   Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params')

In [ ]:
# Train
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)

trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)
trainer.train()

model.save_adapter('./my_adapter')
print('\n✅ Done! Adapter saved to ./my_adapter')

---
*0xticketguy / Harboria Labs | AGPL-3.0*